# Example Validation Workflow

**Phase 1B Infrastructure - Complete Example**

This notebook demonstrates how to use the validation infrastructure built in Phase 1B.1.

**Contents**:
1. Setup and imports
2. Generate synthetic data
3. Run validation
4. Visualize results
5. Parallel simulations
6. Storage and retrieval

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Validation infrastructure
from src.validation import (
    DGPGenerator,
    DGPResult,
    ValidationResult,
    ResultStorage,
    plotting,
    parallel_map,
    parallel_monte_carlo,
)

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Imports successful")

## 2. Generate Synthetic Data

Create a data generating process (DGP) with known treatment effect.

In [ ]:
# Create DGP
dgp = DGPGenerator(
    n=1000,  # Sample size
    p=5,  # Number of confounders
    true_effect=2.0,  # True treatment effect (τ)
    confounding_strength=1.0,  # Confounding strength
    treatment_model="linear",
    outcome_model="linear",
    random_state=42,
)

# Generate single dataset
data = dgp.generate()

print(f"Generated data:")
print(f"  Y (outcome): shape {data.Y.shape}")
print(f"  T (treatment): shape {data.T.shape}")
print(f"  X (confounders): shape {data.X.shape}")
print(f"  True effect (τ): {data.true_effect}")

### Visualize Generated Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Treatment distribution
axes[0].hist(data.T, bins=30, edgecolor="white")
axes[0].set_xlabel("Treatment (T)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Treatment Distribution")

# Outcome distribution
axes[1].hist(data.Y, bins=30, edgecolor="white", color="orange")
axes[1].set_xlabel("Outcome (Y)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Outcome Distribution")

# Treatment vs Outcome
axes[2].scatter(data.T, data.Y, alpha=0.5)
axes[2].set_xlabel("Treatment (T)")
axes[2].set_ylabel("Outcome (Y)")
axes[2].set_title("Treatment vs Outcome")

plt.tight_layout()
plt.show()

## 3. Simple Estimator Example

Demonstrate a simple OLS estimator (biased, but illustrative).

In [ ]:
from sklearn.linear_model import LinearRegression

# Naive OLS (biased due to confounding)
def naive_ols(data: DGPResult) -> float:
    """Naive OLS estimator (biased)."""
    model = LinearRegression()
    X_with_T = np.column_stack([data.T, data.X])
    model.fit(X_with_T, data.Y)
    return model.coef_[0]


# OLS with controls (less biased)
def ols_with_controls(data: DGPResult) -> float:
    """OLS with confounder controls."""
    model = LinearRegression()
    X_with_T = np.column_stack([data.T, data.X])
    model.fit(X_with_T, data.Y)
    return model.coef_[0]


# Estimate
naive_estimate = naive_ols(data)
controlled_estimate = ols_with_controls(data)

print(f"True effect: {data.true_effect:.3f}")
print(f"Naive OLS estimate: {naive_estimate:.3f} (bias: {naive_estimate - data.true_effect:.3f})")
print(
    f"OLS with controls: {controlled_estimate:.3f} (bias: {controlled_estimate - data.true_effect:.3f})"
)

## 4. Monte Carlo Validation

Run multiple simulations to assess bias.

In [ ]:
def run_simulation(sim_id: int, dgp: DGPGenerator) -> dict:
    """Run single Monte Carlo simulation."""
    # Generate data
    data = dgp.generate()

    # Estimate
    estimate = ols_with_controls(data)

    # Calculate bias
    bias = estimate - data.true_effect

    return {"sim_id": sim_id, "estimate": estimate, "bias": bias}


# Run 1000 simulations (sequential for now)
n_simulations = 1000
results = [run_simulation(i, dgp) for i in range(n_simulations)]

# Extract estimates and biases
estimates = np.array([r["estimate"] for r in results])
biases = np.array([r["bias"] for r in results])

print(f"Monte Carlo Results (n={n_simulations}):")
print(f"  Mean estimate: {np.mean(estimates):.3f}")
print(f"  Mean bias: {np.mean(biases):.3f}")
print(f"  Std bias: {np.std(biases):.3f}")
print(f"  RMSE: {np.sqrt(np.mean(biases**2)):.3f}")

### Visualize Bias Distribution

In [ ]:
# Use built-in plotting utility
fig = plotting.plot_bias_distribution(
    biases=biases, true_effect=dgp.true_effect, title="Bias Distribution: OLS with Controls"
)
plt.show()

## 5. Parallel Execution

Use the parallel execution framework for faster simulations.

In [ ]:
# Define simulation function compatible with parallel_monte_carlo
def parallel_simulation(sim_id: int, dgp_params: dict) -> float:
    """Simulation function for parallel execution."""
    # Recreate DGP with offset seed
    dgp_local = DGPGenerator(**dgp_params, random_state=sim_id)
    data = dgp_local.generate()
    return ols_with_controls(data)


# DGP parameters
dgp_params = {
    "n": 1000,
    "p": 5,
    "true_effect": 2.0,
    "confounding_strength": 1.0,
}

# Run in parallel (use 4 cores for demo, 48 in production)
parallel_estimates = parallel_monte_carlo(
    parallel_simulation,
    n_simulations=1000,
    n_jobs=4,
    show_progress=True,
    dgp_params=dgp_params,
)

parallel_biases = np.array(parallel_estimates) - dgp.true_effect

print(f"\nParallel Monte Carlo Results:")
print(f"  Mean bias: {np.mean(parallel_biases):.3f}")
print(f"  RMSE: {np.sqrt(np.mean(parallel_biases**2)):.3f}")

## 6. Storage and Retrieval

Save and load validation results.

In [ ]:
from datetime import datetime

# Create validation result
result = ValidationResult(
    method="OLS_with_controls",
    status="PASS",
    bias=np.mean(biases),
    mse=np.mean(biases**2),
    coverage=0.95,  # Placeholder
    ci_lower=np.percentile(biases, 2.5),
    ci_upper=np.percentile(biases, 97.5),
    n_simulations=n_simulations,
    timestamp=datetime.now(),
    metadata={
        "dgp_n": dgp.n,
        "dgp_p": dgp.p,
        "dgp_true_effect": dgp.true_effect,
        "estimator": "OLS with controls",
    },
)

# Display result
print(result.summary())

In [ ]:
# Save to storage
storage = ResultStorage(base_dir="../results/validation")
filepath = storage.save_result(result, metadata={"notebook": "example_validation"})

print(f"✅ Result saved to: {filepath}")

In [ ]:
# Load results
loaded_results = storage.load_results(limit=5)

print(f"Loaded {len(loaded_results)} results:")
for r in loaded_results:
    print(f"  - {r.method}: {r.status} (bias={r.bias:.3f}, RMSE={np.sqrt(r.mse):.3f})")

In [ ]:
# Get storage summary
summary = storage.get_summary()
print(f"\nStorage Summary:")
print(f"  Total results: {summary['total_results']}")
print(f"  Methods: {summary['methods']}")
print(f"  Statuses: {summary['statuses']}")
print(f"  Cache entries: {summary['cache_entries']}")

## 7. Next Steps

**You've now seen the complete validation workflow!**

To implement your own validation method:

1. Copy `templates/validation_method_template.py`
2. Implement your estimator in `_estimate_effect()`
3. Copy `templates/test_harness_template.py` and write tests
4. Run `pytest` to verify
5. Use `parallel_monte_carlo` for large simulations
6. Generate reports with `plotting` utilities

**See `docs/VALIDATION_IMPLEMENTATION_GUIDE.md` for complete details.**

---

**Generated**: 2025-11-13  
**Phase**: 1B.1 Infrastructure  
**Project**: Double ML Validation Framework  